# **Projeto Final — Aprendizado de Máquina I (2026.2) · Parte II: aplicação**
### Grupo: *Machine Learning Name in progress*

- Tipo de problema: **Classificação**
- Conjunto de dados: [**30000 Spotify Songs**](https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs)
- Objetivo principal: **prever o gênero de uma nova música**


- **Equipe responsável**
    - Ryan Kevin
    - Mateus Bandeira
    - Micaele Brandão
    - Vinicius Flesch
- **Revisão** 
    - Lucas Vieira
    - Mateus Almeida

## 0. Configuração e reprodutibilidade

- Versão da biblioteca: (**Prencher após último commit!!!**)
- Semente global: `67`
- Instalação: `pip install "mlnip-lib @ git+https://github.com/Machine-Learning-Name-in-progress/ML-Library@<hash>"`

### 0.1 Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Biblioteca do grupo
from ml_library.metrics.classification_metrics import (
    accuracy,
    area_under_curve,
    confusion_matrix,
    f1_score,
    log_loss,
    precision,
    recall,
    specificity,
)
from ml_library.models.classification.discriminant_analysis import LDA, QDA
from ml_library.models.classification.knn import KNN
from ml_library.models.classification.logistic_regression import LogisticRegression
from ml_library.models.unsupervised.pca import pca
from ml_library.preprocessing.normalization import normalization
from ml_library.preprocessing.one_hot import one_hot
from ml_library.preprocessing.standardization import standardization
from ml_library.utils._random import set_seed

# TODO: implementar as partes que ainda faltam

### 0.2 Semente e constantes globais

In [ ]:
SEED = 67
set_seed(SEED)
rng = np.random.default_rng(SEED)       # Gerador do notebook (divisão treino/teste)

DATA_PATH = "data/spotify_songs.csv"    # Caminho relativo ao dataset
TARGET = "playlist_genre"               # Coluna que sera prevista

TEST_SIZE = 0.2     # fração para teste (TODO: Justificar no relatório)
N_FOLDS = 5         # k do k-fold

## 1. Problema e dados

Descrição, motivação e licença: **Relatório!!**

- Pergunta:
- Métrica principal:
- Observação (opcional):

### 1.1 Carregamento dos dados

In [ ]:
df = pd.read_csv(DATA_PATH)
df.shape

### 1.2 Dicionário de variáveis

| Variável | Tipo | Unidade | Descrição |
|---|---|---|---|
| `track_id`, `track_album_id`, `playlist_id` | identificador | — | IDs do Spotify |
| `track_name`, `track_artist`, `track_album_name`, `playlist_name` | texto | — | nomes |
| `track_album_release_date` | data | — | lançamento do álbum: AAAA-MM-DD, AAAA-MM ou só AAAA |
| `track_popularity` | numérica | 0–100 | popularidade da faixa no Spotify |
| `playlist_genre` | categórica | — | gênero da playlist (6 classes) |
| `playlist_subgenre` | categórica | — | subgênero da playlist |
| `danceability`, `energy`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence` | numérica | 0–1 | atributos de áudio calculados pelo Spotify |
| `key` | categórica | 0–11 | tom |
| `mode` | binária | — | 1 = maior, 0 = menor |
| `loudness` | numérica | dB | volume médio da faixa |
| `tempo` | numérica | BPM | andamento estimado |
| `duration_ms` | numérica | ms | duração |

## 2. Análise exploratória (EDA)

### 2.1 Dimensões, tipos e primeiras linhas

In [ ]:
# Grupos de colunas
ID_COLS = ["track_id", "track_album_id", "playlist_id"]
TEXT_COLS = ["track_name", "track_artist", "track_album_name", "playlist_name"]
DATE_COL = "track_album_release_date"
CAT_COLS = ["key", "mode"]
NUM_COLS = [
    "track_popularity", "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration_ms",
]

print(f"{df.shape[0]} linhas x {df.shape[1]} colunas")
display(df.dtypes.astype(str).value_counts().rename("número de colunas").to_frame())
display(df.head())

**Linhas repetidas.** A mesma faixa aparece em várias playlists, então `track_id` se repete — às vezes com gênero diferente.

In [ ]:
genres_per_track = df.groupby("track_id")[TARGET].nunique()
unique_tracks = df.drop_duplicates("track_id")

print(f"Linhas idênticas: {df.duplicated().sum()}")
print(f"Faixas únicas (track_id): {df['track_id'].nunique()} de {len(df)} linhas")
print(f"Linhas com track_id repetido: {len(df) - df['track_id'].nunique()}")
print(f"Faixas em mais de um gênero: {(genres_per_track > 1).sum()}")

# Mesmo nome + artista com track_id diferente (single vs. álbum, relançamentos)
print(f"Mesmo nome+artista, outro track_id:  {unique_tracks.duplicated(['track_name', 'track_artist']).sum()}")

genres_per_track.value_counts().sort_index().rename_axis("gêneros por faixa").rename("numero de faixas").to_frame()

### 2.2 Valores ausentes

Quantos NaN por coluna? Faltam ao acaso ou há padrão? O que um NA representa?

In [ ]:
missing = df.isna().sum()
display(missing[missing > 0].rename("ausentes").to_frame())
df[df.isna().any(axis=1)][["track_id", "track_name", "track_artist", "track_album_name", TARGET]]

**Faltantes disfarçados.** Zeros e valores impossíveis que podem estar no lugar de "sem informação".

In [ ]:
print(f"tempo == 0: {(df['tempo'] == 0).sum()}")
print(f"duration_ms < 30 s: {(df['duration_ms'] < 30000).sum()}")
print(f"instrumentalness == 0: {(df['instrumentalness'] == 0).mean():.1%}")
print(f"track_popularity == 0: {(df['track_popularity'] == 0).mean():.1%}")
display((df["track_popularity"] == 0).groupby(df[TARGET]).mean().round(3).rename("fração com popularidade 0").to_frame())

df[df["tempo"] == 0][["track_name", "track_artist"] + NUM_COLS]

- Valores ausentes só em 5 linhas, e só em colunas de texto (`track_name`, `track_artist`, `track_album_name`), que não entram no modelo.
- Uma faixa tem `tempo`, `danceability`, `valence` etc. iguais a 0 e dura 4 s
- `instrumentalness == 0` em ~37% é valor real (faixa com voz), não faltante.
- `track_popularity == 0` não é aleatório: o dobro em `edm` (~12%) em relação a `pop` (~6%).

### 2.3 Distribuição da variável-alvo

- Regressão: histograma, assimetria, outliers.
- Classificação: contagem por classe, desbalanceamento.

In [ ]:
counts = pd.DataFrame({
    "linhas": df[TARGET].value_counts(),
    "faixas únicas": unique_tracks[TARGET].value_counts(),
})
props = counts / counts.sum()
display(pd.concat([counts, props.round(3).add_suffix(" (prop.)")], axis=1))
print(f"Razão maior/menor classe (faixas únicas): {counts['faixas únicas'].max() / counts['faixas únicas'].min():.2f}")

ax = props.plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("proporção")
ax.set_title(f"Distribuição de {TARGET}")
ax.legend()
plt.show()

**Rótulos conflitantes.** Faixas que aparecem em mais de um gênero: quais pares de gêneros dividem faixas? (diagonal = número de faixas multigênero daquele gênero)

In [ ]:
multi = df[df["track_id"].isin(genres_per_track[genres_per_track > 1].index)]

# one_hot marca o gênero de cada linha; o max por faixa junta todas as playlists da mesma faixa
encoded = one_hot(multi[["track_id", TARGET]], [TARGET], drop=True)
genre_onehot = encoded.groupby("track_id").max()
genre_onehot.columns = genre_onehot.columns.str.removeprefix(f"{TARGET}_").str.removesuffix("_onehot")
genre_onehot = genre_onehot.sort_index(axis=1)

multi_class = genre_onehot.T @ genre_onehot
multi_class = multi_class.mask(np.eye(len(multi_class), dtype=bool), 0)

multi_class

- Classes quase balanceadas: 15–18% por linha, 15–19% por faixa única; razão maior/menor ≈ 1,3.
- Deduplicar muda a ordem: `edm` é a maior classe nas linhas, `rap` nas faixas únicas.
- Pares que mais dividem faixas: `pop`–`edm`, `pop`–`r&b`, `pop`–`latin`. `rock` quase não divide.

### 2.4 Distribuição das variáveis explicativas

- Numéricas: histogramas/boxplots.
- Categóricas: contagens.

In [ ]:
release_year = df[DATE_COL].str[:4].astype(int)
num_data = df[NUM_COLS].assign(release_year=release_year)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flat, num_data.columns):
    ax.hist(num_data[col], bins=50, color="tab:blue", alpha=0.8)
    ax.set_title(f"{col}  (assimetria {num_data[col].skew():.2f})", fontsize=10)
fig.suptitle("Variáveis numéricas")
fig.tight_layout()
plt.show()

In [ ]:
q1, q3 = num_data.quantile(0.25), num_data.quantile(0.75)
iqr = q3 - q1
outliers = ((num_data < q1 - 1.5 * iqr) | (num_data > q3 + 1.5 * iqr)).mean()

summary = num_data.describe().T
summary["assimetria"] = num_data.skew()
summary["% outliers (IQR)"] = (100 * outliers).round(1)
summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 3.5))

# Os valores de key incluem os semitons também (achei curioso não sabia que existia - Vinicius)
df["key"].value_counts().sort_index().plot.bar(ax=axes[0], rot=0, title="key (0 = Dó … 11 = Si)")
df["mode"].value_counts().sort_index().plot.bar(ax=axes[1], rot=0, title="mode (0 = menor, 1 = maior)")
fig.tight_layout()
plt.show()

- Escalas muito diferentes: `duration_ms` ~10⁵, `tempo` ~10², `loudness` em dB negativo, atributos de áudio em 0–1.
- Assimetria forte à direita: `instrumentalness` (2,8), `liveness` (2,1), `speechiness` (2,0), `acousticness` (1,6). À esquerda: `loudness` (−1,4), mínimo −46 dB.
- `instrumentalness` tem 21% de outliers pelo IQR só porque a mediana é ~0: a regra do IQR não serve aqui.
- `track_popularity` tem pico em 0 (8% das linhas) separado do resto da distribuição.
- `release_year` vai de 1957 a 2020, concentrado depois de 2010.
- `key` e `mode` sem categoria rara; `mode` = 1 em 57%.

### 2.5 Relação entre variáveis

Correlação entre preditores. Relação com o alvo: dispersão (regressão) ou boxplot por classe (classificação).

In [ ]:
corr = num_data.assign(key=df["key"], mode=df["mode"]).corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.columns)
corr_values = corr.to_numpy()   # array numpy: tipo numérico claro para o Pylance
for i in range(len(corr)):
    for j in range(len(corr)):
        r = corr_values[i, j]
        ax.text(j, i, f"{r:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(r) > 0.5 else "black")
fig.colorbar(im, ax=ax)
ax.set_title("Correlação de Pearson entre preditores")
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
pairs.sort_values(key=abs, ascending=False).head(5).rename("r").to_frame()

In [ ]:
genres = sorted(df[TARGET].unique())

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flat, num_data.columns):
    ax.boxplot([num_data.loc[df[TARGET] == g, col] for g in genres], tick_labels=genres, showfliers=False)
    ax.set_title(col, fontsize=10)
    ax.tick_params(axis="x", labelsize=8)
fig.suptitle("Variáveis numéricas por gênero (sem outliers)")
fig.tight_layout()
plt.show()

**Poder de separação de cada variável.** η² = variância entre classes / variância total (0 = médias iguais em todos os gêneros; 1 = separa perfeitamente). Olha só a média, então é um ranking aproximado.

In [ ]:
def eta_squared(x, y):
    """Fração da variância de x explicada pela média de cada classe de y."""
    return ((x.groupby(y).transform("mean") - x.mean()) ** 2).sum() / ((x - x.mean()) ** 2).sum()


eta = num_data.assign(mode=df["mode"]).apply(lambda col: eta_squared(col, df[TARGET])).sort_values()
ax = eta.plot.barh(figsize=(7, 5), title="η² de cada variável em relação ao gênero")
ax.set_xlabel("η²")
plt.show()

# key é nominal: comparar a distribuição por gênero em vez da média
pd.crosstab(df[TARGET], df["key"], normalize="index").round(3)

**Projeção PCA.** `pca` da biblioteca sobre as numéricas padronizadas (`key` fica de fora por ser nominal). Mostra se os gêneros se separam com combinações lineares das variáveis.

In [ ]:
eda_pca = pca(n_components=num_data.shape[1], standardize=True)
scores = eda_pca.fit_transform(num_data.to_numpy())

# Com standardize=True cada coluna tem variância 1, então a variância total é o nº de colunas
explained = scores.var(axis=0) / num_data.shape[1]
pc_names = [f"PC{i + 1}" for i in range(len(explained))]
display(pd.DataFrame({"variância explicada": explained, "acumulada": explained.cumsum()}, index=pc_names).round(3).T)
display(pd.DataFrame(eda_pca.components[:, :2], index=num_data.columns, columns=pc_names[:2]).round(2))

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
for ax, g in zip(axes.flat, genres):
    is_g = (df[TARGET] == g).to_numpy()
    ax.scatter(scores[~is_g, 0], scores[~is_g, 1], s=2, c="lightgray", alpha=0.3, rasterized=True)
    ax.scatter(scores[is_g, 0], scores[is_g, 1], s=2, alpha=0.4, rasterized=True)
    ax.set_title(g)
for ax in axes[1]:
    ax.set_xlabel(f"PC1 ({explained[0]:.0%})")
for ax in axes[:, 0]:
    ax.set_ylabel(f"PC2 ({explained[1]:.0%})")
fig.suptitle("Projeção nas 2 primeiras componentes principais (cinza = demais gêneros)")
fig.tight_layout()
plt.show()

**Vazamento.** Colunas que vêm da playlist (e não da música) e quanto delas determina o gênero sozinha.

In [ ]:
for col in ["playlist_subgenre", "playlist_name", "playlist_id", "track_album_id"]:
    single_genre = (df.groupby(col)[TARGET].nunique() == 1).mean()
    print(f"{col:18s} {df[col].nunique():6d} valores; {single_genre:6.1%} mapeiam para um único gênero")

pd.crosstab(df["playlist_subgenre"], df[TARGET])

- Correlações moderadas: `energy`–`loudness` (0,68), `energy`–`acousticness` (−0,54). Nenhum par acima de 0,7.
- Mais discriminantes (η²): `release_year` (0,30), `speechiness` (0,20), `danceability` (0,19), `energy` (0,14). Quase nada: `liveness`, `mode`, `key`.
- `rock`: mais antigo (mediana 2000), menos dançante, menos `speechiness`. `rap`: `speechiness` alto. `edm`: mais `energy` e `instrumentalness`.
- η² só compara médias: `tempo` tem η² baixo, mas no `edm` é muito concentrado (~120–135 BPM). Modelos que usam a dispersão por classe (QDA, KNN) podem aproveitar isso.
- `playlist_subgenre` determina o gênero em 100% dos casos; `playlist_name`/`playlist_id` em ~99%.
- PCA: PC1 ≈ energia/volume × acústico (separa `edm` de `r&b`); PC2 ≈ faixa longa e antiga × dançante e falada (isola `rock`, oposto a `rap`/`latin`). As duas explicam só 32% da variância e `pop` fica no meio de todos.
- São precisas 10 das 12 componentes para 90% da variância: pouca redundância entre as variáveis.

### 2.6 Conclusões da EDA


- O que tratar no pré-processamento:
    - **Deduplicar por `track_id`** (na 3.1, antes da divisão). `df.duplicated()` encontra 0 linhas: não basta.
    - Faixas em mais de um gênero (~1,7 mil): decidir entre manter a 1ª ocorrência ou descartar.
    - Remover a faixa com `tempo == 0` (análise de áudio inválida).
    - Remover identificadores e textos (`ID_COLS`, `TEXT_COLS`) e `playlist_subgenre`.
    - `track_album_release_date` → `release_year` (é a variável mais discriminante).
    - `key` nominal: one-hot ou descartar (η² ≈ 0). `mode` binária: manter como está.
    - Padronizar as numéricas (escalas de 0–1 a ~10⁵; KNN, regressão logística e rede neural são sensíveis).
    - Opcional: `log1p` em `speechiness`, `acousticness`, `instrumentalness`, `liveness` (assimetria > 1,5).
    - Reduzir dimensão com PCA não deve compensar (10 de 12 componentes para 90% da variância).
- Riscos (vazamento, colinearidade, outliers, desbalanceamento):
    - **Vazamento por duplicata**: 32 833 linhas, 28 356 faixas. A mesma faixa em treino e teste infla a métrica. Há ainda ~2,1 mil pares com mesmo nome+artista e `track_id` diferente (relançamentos).
    - **Vazamento por coluna**: `playlist_subgenre` (100%), `playlist_name`/`playlist_id` (~99%) entregam o gênero.
    - **Vazamento por álbum** (mais sutil): 92% dos álbuns têm um só gênero; um split aleatório põe faixas do mesmo álbum nos dois lados.
    - **Rótulo ruidoso**: o mesmo áudio com gêneros diferentes limita a acurácia máxima; esperar confusão entre `pop`, `edm`, `latin` e `r&b`.
    - Colinearidade moderada (`energy`–`loudness` 0,68). Pesa mais na QDA (covariância por classe) que no KNN.
    - Outliers reais (não erros): `loudness` até −46 dB, caudas longas nos atributos 0–1.
    - Desbalanceamento leve (razão ≈ 1,3): estratificar a divisão e reportar F1-macro junto com a acurácia.

## 3. Divisão treino/teste

### 3.1 Remoção de linhas duplicadas

In [ ]:
# Quantos gêneros distintos cada faixa tem
n_genres = df.groupby("track_id")[TARGET].transform("nunique")

# Usa apenas faixas que possuem 1 único gênero
df_clean = df[n_genres == 1]

# Remove as duplicatas de um único gênero por faixa
df_clean = df_clean.drop_duplicates("track_id")

# Remove faixas com aúdio inválido
df_clean = df_clean[df_clean["tempo"] > 0]

# Organiza os índices para 0,1,2,...
df_clean = df_clean.reset_index(drop=True)

print(f"Todas as faixas são únicas? {df_clean["track_id"].is_unique}")

rows_df = len(df)
rows_clean_df = len(df_clean)

print(f"{rows_df} linhas -> {rows_clean_df} faixas -> {rows_df - rows_clean_df} removidas")

### 3.2 Divisão

Separação em **dados de treino** e **dados de teste**

In [ ]:
# Sorteia uma fração dos dados (separar dados de teste)
test_df = df_clean.groupby(TARGET).sample(frac=TEST_SIZE, random_state=SEED)

# Remove os dados que estão na fatia de teste
# O que sobra será usado como dados de treino
train_df = df_clean.drop(test_df.index)

# Embaralha as linhas
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Ordena os índices para 0,1,2,...
test_df = test_df.reset_index(drop=True)

### 3.3 Conferência

Tamanho final dos conjuntos de teste e treino

In [ ]:
print(f"Tamanho dos dados originais: {len(df_clean)}")
print(f"Tamanho dos dados de treino (fração {1 - TEST_SIZE}): {len(train_df)}")
print(f"Tamanho dos dados de teste (fração {TEST_SIZE}): {len(test_df)}")

## 4. Pré-processamento

Todo parâmetro (média, desvio, categorias do one-hot) é calculado **só no treino** e aplicado no outro conjunto. As subseções registram as decisões; a 4.6 junta tudo numa função, reutilizada em cada fold (seção 6) e no teste (seção 8).

### 4.1 Remoção de colunas

Colunas que não serão usadas:

| Coluna | Motivo |
|---|---|
| `track_id`, `track_album_id`, `playlist_id` | identificadores do Spotify; irrelevantes |
| `track_name`, `track_artist`, `track_album_name` | texto livre |
| `playlist_name` | texto |
| `playlist_subgenre` | vazamento: cada subgênero pertence a um único gênero (100%) |
| `track_album_release_date` | substituída por `release_year` (numérica) |

In [ ]:
# Colunas que não vão ser usadas no modelo
DROP_COLS = ID_COLS + TEXT_COLS + ["playlist_subgenre", DATE_COL]

# Colunas que precisam ser colocadas na mesma escala
SCALE_COLS = NUM_COLS + ["release_year"]

# Função que deixa apenas colunas que o modelo pode usar:
#   1. Converte "release_year" para apenas ano (numérico)
#   2. Trata a coluna "key" como categoria (evita ordenação)
#   3. Apaga as colunas definidas acima

def basic_transform(data):
    out = data.copy()
    
    # 1 cria a coluna "year"
    out["release_year"] = out[DATE_COL].str[:4].astype(int)
    
    # 2 converte "key" para categórica
    out["key"] = out["key"].astype(str)
    
    # 3 remove as colunas necessárias
    out = out.drop(columns=DROP_COLS)
    return out

### 4.2 Valores ausentes

Confirmação de que não existem valores vazios

In [ ]:
print(basic_transform(train_df).isna().sum().sum() != 0)

### 4.3 Função de pré-processamento e matrizes finais

In [ ]:
def fit_preprocess(train):
    
    # Aplica a transformação de 4.1 nos dados fornecidos
    data = basic_transform(train)
    
    # Aplica codificação one-hot na coluna "key" (notas)
    encoded = one_hot(data, ["key"], drop=True)
    
    # Ordena as colunas "_onehot" que foram criadas na codificação
    onehot_cols = sorted(c for c in encoded.columns if c.endswith("_onehot"))
    
    # Ordena a primeira coluna do onehot
    # "key_0_onehot" vira categoria de referência: definida a partir das outras
    onehot_cols = onehot_cols[1:]
    
    # Usa a função de padronização, salvando apenas o dicionário com
    # média e desvio de cada coluna
    _, scale_params = standardization(data, SCALE_COLS)
    
    # Retorna as colunas do Onehot e os parâmetros de escala (média e desvio)
    return {"onehot_cols": onehot_cols, "scale": scale_params}

## 5. Modelos candidatos

| Modelo | Classe/função da biblioteca | Hiperparâmetros | Por que testar |
|---|---|---|---|
| Baseline |  | — | referência mínima |
|  |  |  |  |
| Rede neural (obrigatória) |  |  |  |

### 5.1 Baseline

### 5.2 Modelo 1

### 5.3 Modelo 2

### 5.4 Modelo 3

## 6. Validação e seleção de hiperparâmetros

Usar **só o conjunto de treino**. O teste não aparece nesta seção.

### 6.1 Função de validação

### 6.2 Busca de hiperparâmetros — por modelo

## 7. Comparação de modelos

| Modelo | Melhores hiperparâmetros | Métrica principal |
|---|---|---|
|  |  |  |

### 7.1 Escolha do modelo final

Texto.

## 8. Avaliação final no conjunto de teste

Cuidado pra não usar o treino.

### 8.1 Treino final

### 8.2 Métricas no teste

### 8.3 Gráficos

## 9. Conclusão

Tópicos curtos; detalhes e limitações no relatório.

- Melhor modelo e desempenho no teste:
- Principais limitações (opcional):